# FLUX.1 Krea [dev] — 조선 스타일 LoRA 학습 (Kaggle)

데이터셋 **197쌍**(joseon 트리거) → ai-toolkit(ostris)로 Krea dev LoRA 학습.

**실행 전 체크리스트**
1. 우측 **Settings → Accelerator = `GPU T4 x2`** (또는 P100), **Internet = On**
2. HuggingFace에서 `black-forest-labs/FLUX.1-Krea-dev` **라이선스 동의**(게이트 모델)
3. 우측 **Add-ons → Secrets** 에 `HF_TOKEN` 추가(HF 토큰, read 권한)
4. 우측 **Input → Add Input** 으로 업로드한 데이터셋(zip 포함) 추가
5. 위에서부터 셀 순서대로 실행


## 1. GPU 확인


In [ ]:
!nvidia-smi

## 2. ai-toolkit 설치 (약 3~5분)


In [ ]:
import os
os.chdir('/kaggle/working')
!git clone https://github.com/ostris/ai-toolkit.git
os.chdir('/kaggle/working/ai-toolkit')
!git submodule update --init --recursive
# torch는 Kaggle에 이미 설치됨. 나머지 의존성만 설치
!pip install -q -r requirements.txt
!pip install -q optimum-quanto
print('설치 완료')

## 3. HuggingFace 로그인 (Secrets의 HF_TOKEN 사용)


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
tok = UserSecretsClient().get_secret('HF_TOKEN')
login(token=tok)
print('HF 로그인 OK')

## 4. 데이터셋 준비
업로드한 Kaggle Input에서 jpg/txt 페어를 찾아 `/kaggle/working/dataset` 으로 모음.
(zip으로 올렸으면 자동 압축해제, 폴더로 올렸으면 그대로 복사)


In [ ]:
import os, glob, zipfile, shutil
DST = '/kaggle/working/dataset'
os.makedirs(DST, exist_ok=True)

# 4-1) /kaggle/input 안의 zip 먼저 풀기
for z in glob.glob('/kaggle/input/**/*.zip', recursive=True):
    print('unzip:', z)
    with zipfile.ZipFile(z) as zf:
        zf.extractall('/kaggle/working/_unzipped')

# 4-2) input + 방금 푼 폴더에서 jpg/txt 페어 수집
roots = ['/kaggle/input', '/kaggle/working/_unzipped']
jpgs = {}
for r in roots:
    for p in glob.glob(os.path.join(r,'**','*.jpg'), recursive=True):
        jpgs[os.path.splitext(os.path.basename(p))[0]] = p
txts = {}
for r in roots:
    for p in glob.glob(os.path.join(r,'**','*.txt'), recursive=True):
        txts[os.path.splitext(os.path.basename(p))[0]] = p

n=0
for h, jp in jpgs.items():
    if h in txts:
        shutil.copy(jp, os.path.join(DST, h+'.jpg'))
        shutil.copy(txts[h], os.path.join(DST, h+'.txt'))
        n+=1
print(f'데이터셋 페어 {n}쌍 준비 → {DST}')
assert n>0, '페어를 못 찾음! Input에 데이터셋을 추가했는지 확인하세요.'

## 5. 학습 설정(config) 작성
rank/steps/resolution 등을 바꾸려면 이 셀에서 수정.


In [ ]:
config_yaml = r'''
---
job: extension
config:
  name: joseon_krea_lora
  process:
    - type: sd_trainer
      training_folder: /kaggle/working/output
      device: cuda:0
      trigger_word: joseon
      network:
        type: lora
        linear: 16
        linear_alpha: 16
      save:
        dtype: float16
        save_every: 500
        max_step_saves_to_keep: 4
      datasets:
        - folder_path: /kaggle/working/dataset
          caption_ext: txt
          caption_dropout_rate: 0.05
          cache_latents_to_disk: true
          resolution: [768]
      train:
        batch_size: 1
        steps: 2500
        gradient_accumulation_steps: 1
        train_unet: true
        train_text_encoder: false
        gradient_checkpointing: true
        noise_scheduler: flowmatch
        optimizer: adamw8bit
        lr: 1e-4
        skip_first_sample: true
        dtype: bf16
        ema_config:
          use_ema: false
      model:
        name_or_path: black-forest-labs/FLUX.1-Krea-dev
        is_flux: true
        quantize: true
        low_vram: true
      sample:
        sampler: flowmatch
        sample_every: 500
        width: 768
        height: 768
        prompts:
          - "joseon, 16th-century Joseon Korea, a weary thin peasant in worn hanbok by a thatched house at dusk, cold blue twilight"
          - "joseon, ancient Baekje Korea, a hillside lined with earthen royal tombs, muted cold tones"
        neg: ""
        seed: 42
        walk_seed: true
        guidance_scale: 4
        sample_steps: 20
meta:
  name: joseon_krea_lora
  version: '1.0'
'''
with open('/kaggle/working/config.yaml','w') as f:
    f.write(config_yaml)
print('config.yaml 작성 완료')

## 6. 학습 실행
T4 기준 2500스텝 ≈ **3~5시간**. 로그에 loss / 500스텝마다 샘플 이미지가 나옴.

> **T4에서 bf16 관련 에러가 나면** 위 config 셀의 `dtype: bf16` → `dtype: float16` 으로 바꾸고 5·6번 셀 재실행.


In [ ]:
import os
os.chdir('/kaggle/working/ai-toolkit')
!python run.py /kaggle/working/config.yaml

## 7. 결과물 다운로드용 압축
`joseon_krea_lora.safetensors` 가 최종 LoRA. 우측 **Output** 패널에서 다운로드하거나 아래 zip 사용.


In [ ]:
import shutil, glob
out='/kaggle/working/output'
for f in glob.glob(out+'/**/*.safetensors', recursive=True):
    print(f)
shutil.make_archive('/kaggle/working/joseon_krea_lora_result','zip',out)
print('→ /kaggle/working/joseon_krea_lora_result.zip 생성(Output에서 다운로드)')